# Entrainement final / Random Forest

Ce notebook lit la meilleure configuration produite par `hyperparameter_search.py`, recharge le preprocessing centralisé avec `augment=False`, entraine le Random Forest sur tout le pool train+validation, puis sauvegarde les artefacts binaires dans `models/random_forest/final_model/` et les métadonnées JSON dans `results/random_forest/final_model/`.

Le jeu de test final n'est pas touché  ici.

In [1]:
import gc, json, sys
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
while not (ROOT / "cross_validation.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ipynb.fs.full.preprocessing import get_data_pipeline
from random_forest.hyperparameter_search import extract_hog_features, HOG_CONFIG, LABEL_NAMES, SEED

MODEL_DIR = ROOT / "models" / "random_forest" / "final_model"
RESULTS_DIR = ROOT / "results" / "random_forest" / "final_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = ROOT / "results" / "random_forest" / "hyperparameter_search" / "best_config.json"

c:\Users\admin\git\repository\Zoidberg2.0\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Lecture de la meilleure configuration

In [2]:
with open(BEST_PATH, encoding="utf-8") as f:
    best = json.load(f)
cfg = best["best_config"]
print("Meilleure configuration :")
print(json.dumps(cfg, indent=2))
print("\nMétriques CV associées :")
print(json.dumps(best["best_metrics"], indent=2))

Meilleure configuration :
{
  "n_estimators": 300,
  "max_depth": null,
  "min_samples_split": 5,
  "min_samples_leaf": 1,
  "max_features": "sqrt",
  "criterion": "gini",
  "use_smote": true,
  "smote_k_neighbors": 3
}

Métriques CV associées :
{
  "accuracy_mean": 0.764107658011632,
  "precision_mean": 0.7557050495894301,
  "recall_mean": 0.7490818432313667,
  "f1_mean": 0.7470254996335886
}


## 2. Préparation des données sans augmentation

In [3]:
pipeline = get_data_pipeline(augment=False)
train_view = pipeline["train_pool_train_view"]
print(f"augment = {pipeline['augment']}")
print(f"Pool d'entraînement final : {len(train_view)} images")

X_train, y_train = extract_hog_features(train_view, desc="Train final HOG")
print(f"Features : {X_train.shape}")
print(f"Distribution : {np.bincount(y_train)}")

Using the latest cached version of the dataset since PAR8/chest-xray-pneumonia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\admin\.cache\huggingface\datasets\PAR8___chest-xray-pneumonia\default\0.0.0\42d3b32e6fc8f1c1974fd14f23bb49e7a130b801 (last modified on Thu Mar  5 10:49:38 2026).
c:\Users\admin\git\repository\Zoidberg2.0\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


augment = False
Pool d'entraînement final : 5227 images


Features : (5227, 34992)
Distribution : [1349 2536 1342]


## 3. Standardisation, équilibrage et entraînement

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

if cfg["use_smote"]:
    smote = SMOTE(random_state=SEED, k_neighbors=cfg["smote_k_neighbors"])
    X_fit, y_fit = smote.fit_resample(X_train_scaled, y_train)
else:
    X_fit, y_fit = X_train_scaled, y_train

model = RandomForestClassifier(
    n_estimators=cfg["n_estimators"],
    max_depth=cfg["max_depth"],
    min_samples_split=cfg["min_samples_split"],
    min_samples_leaf=cfg["min_samples_leaf"],
    max_features=cfg["max_features"],
    criterion=cfg["criterion"],
    random_state=SEED,
    n_jobs=-1,
    class_weight=cfg["class_weight"],
    verbose=1,
)
model.fit(X_fit, y_fit)
print("Entraînement terminé.")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   11.9s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:   56.7s


Entraînement terminé.


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  1.5min finished


## 4. Sauvegarde du modèle final

In [ ]:
model_path = MODEL_DIR / "random_forest_final.joblib"
scaler_path = MODEL_DIR / "scaler.joblib"
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

meta = {
    "model": "RandomForestClassifier",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "config": cfg,
    "augmentation": False,
    "feature_type": "HOG",
    "hog_config": HOG_CONFIG,
    "train_pool_size": int(len(train_view)),
    "fit_samples": int(len(y_fit)),
    "n_features": int(X_train.shape[1]),
    "label_names": LABEL_NAMES,
    "model_path": str(model_path),
    "scaler_path": str(scaler_path),
}
with open(RESULTS_DIR / "train_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Modèle : {model_path}")
print(f"Scaler : {scaler_path}")
print(f"Meta   : {RESULTS_DIR / 'train_meta.json'}")

del model, scaler
gc.collect()

Mod?le : C:\Users\admin\git\repository\Zoidberg2.0\models\random_forest\final_model\random_forest_final.joblib
Scaler : C:\Users\admin\git\repository\Zoidberg2.0\models\random_forest\final_model\scaler.joblib
M?ta   : C:\Users\admin\git\repository\Zoidberg2.0\results\random_forest\final_model\train_meta.json


102